# Phase 1 Slice C: 2D Rendering, Pan/Zoom, Slab Modes

This notebook validates the new `POST /render/image` flow and navigation helpers (`set-plane`, `pan`, `zoom`) end to end.

It uses a tiny deterministic OME-Zarr dataset so we can assert orientation, slab behavior, and state invariants.


In [1]:
from __future__ import annotations

import base64
import tempfile
from io import BytesIO
from pathlib import Path

import numpy as np
import zarr
from fastapi.testclient import TestClient
from PIL import Image

from lucida.client import LucidaClient
from lucida.server.app import create_app
from lucida.service.dataset_service import DatasetService


def create_render_omezarr(uri: str) -> str:
    root = zarr.open_group(store=uri, mode="w")
    shape_level0 = (1, 3, 4, 5, 6)
    data_level0 = np.zeros(shape_level0, dtype=np.uint16)
    for c in range(shape_level0[1]):
        for z in range(shape_level0[2]):
            for y in range(shape_level0[3]):
                for x in range(shape_level0[4]):
                    data_level0[0, c, z, y, x] = np.uint16((c * 1000) + (z * 100) + (y * 10) + x)
    data_level1 = data_level0[:, :, ::2, ::2, ::2]

    root.create_array("0", data=data_level0, chunks=(1, 1, 2, 3, 3), overwrite=True)
    root.create_array("1", data=data_level1, chunks=(1, 1, 1, 2, 2), overwrite=True)

    root.attrs["multiscales"] = [
        {
            "name": "primary",
            "axes": [
                {"name": "t", "type": "t"},
                {"name": "c", "type": "c"},
                {"name": "z", "type": "z"},
                {"name": "y", "type": "y"},
                {"name": "x", "type": "x"},
            ],
            "datasets": [
                {"path": "0", "coordinateTransformations": [{"type": "scale", "scale": [1, 1, 1, 1, 1]}]},
                {"path": "1", "coordinateTransformations": [{"type": "scale", "scale": [1, 1, 2, 2, 2]}]},
            ],
        }
    ]
    root.attrs["omero"] = {
        "channels": [
            {"index": 0, "label": "c0", "color": "ffffff", "window": {"start": 0, "end": 500}},
            {"index": 1, "label": "c1", "color": "ff0000", "window": {"start": 0, "end": 1500}},
            {"index": 2, "label": "c2", "color": "00ff00", "window": {"start": 0, "end": 2500}},
        ]
    }
    return uri


def decode_rgba(b64: str) -> np.ndarray:
    raw = base64.b64decode(b64)
    return np.asarray(Image.open(BytesIO(raw)).convert("RGBA"), dtype=np.uint8)


## Step 1 - Open dataset and create a 2D view


In [2]:
tmp_dir = Path(tempfile.mkdtemp(prefix="lucida-render-"))
dataset_uri = create_render_omezarr(str(tmp_dir / "render.zarr"))

service = DatasetService()
app = create_app(dataset_service=service)
http_client = TestClient(app)
client = LucidaClient(client=http_client)

session = client.create_session()
opened = client.open_dataset(uri=dataset_uri, session_id=session.session_id)
created = client.create_view(dataset_id=opened.dataset_summary.dataset_id, session_id=session.session_id, mode="2d")

assert opened.schema_version == 1
assert created.view_state.view_id
roles = {axis.role for axis in opened.dataset_summary.axes}
assert {"x", "y", "z"}.issubset(roles)
created.view_state.view_id


'view_6a5892efa3f24731'

## Step 2 - Configure predictable rendering and test XY/XZ/YZ


In [3]:
view_id = created.view_state.view_id
client.update_view(
    view_id=view_id,
    session_id=session.session_id,
    patch=[
        {
            "op": "replace",
            "path": "/layers/0/image",
            "value": {
                "channel_mode": "single",
                "channels": [
                    {
                        "index": 0,
                        "enabled": True,
                        "color_rgba": [1.0, 1.0, 1.0, 1.0],
                        "contrast": {"policy": "fixed", "min": 0.0, "max": 500.0, "p_low": 1.0, "p_high": 99.0},
                        "gamma": 1.0,
                    }
                ],
                "interpolation": "nearest",
            },
        },
        {"op": "replace", "path": "/view_2d/camera/zoom", "value": 1.0},
        {"op": "replace", "path": "/view_2d/camera/center_world", "value": [3.0, 2.0]},
        {"op": "replace", "path": "/view_2d/slice", "value": {"axis": "z", "index": 2, "slab": {"thickness_vox": 1, "mode": "single"}}},
    ],
)

xy = client.render_image(view_id=view_id, session_id=session.session_id, width_px=6, height_px=5)
xy_img = decode_rgba(xy.images[0].bytes_base64)
assert xy.schema_version == 1
assert xy.status == "ok"
assert xy.images[0].mime == "image/png"
assert xy_img.shape == (5, 6, 4)

client.set_plane(view_id=view_id, plane="xz", session_id=session.session_id)
xz = client.render_image(view_id=view_id, session_id=session.session_id, width_px=6, height_px=4)
xz_img = decode_rgba(xz.images[0].bytes_base64)
assert xz_img.shape == (4, 6, 4)

client.set_plane(view_id=view_id, plane="yz", session_id=session.session_id)
yz = client.render_image(view_id=view_id, session_id=session.session_id, width_px=5, height_px=4)
yz_img = decode_rgba(yz.images[0].bytes_base64)
assert yz_img.shape == (4, 5, 4)

{
    "xy_shape": xy_img.shape,
    "xz_shape": xz_img.shape,
    "yz_shape": yz_img.shape,
}


{'xy_shape': (5, 6, 4), 'xz_shape': (4, 6, 4), 'yz_shape': (4, 5, 4)}

## Step 3 - Pan/zoom helpers, slab modes, and state invariants


In [4]:
before = client.get_view(view_id=view_id, session_id=session.session_id).view_state
client.pan(view_id=view_id, session_id=session.session_id, dx_px=12.0, dy_px=-8.0)
client.zoom(view_id=view_id, session_id=session.session_id, factor=0.75)

client.update_view(
    view_id=view_id,
    session_id=session.session_id,
    patch=[
        {"op": "replace", "path": "/view_2d/plane", "value": "xy"},
        {"op": "replace", "path": "/view_2d/camera/center_world", "value": [3.0, 2.0]},
        {"op": "replace", "path": "/view_2d/camera/zoom", "value": 1.0},
        {"op": "replace", "path": "/selectors", "value": [{"axis": "z", "kind": "range", "start": 1, "end_exclusive": 4, "clamp": True}]},
    ],
)

single = client.update_view(
    view_id=view_id,
    session_id=session.session_id,
    patch=[{"op": "replace", "path": "/view_2d/slice/slab", "value": {"thickness_vox": 5, "mode": "single"}}],
)
single_render = client.render_image(view_id=view_id, session_id=session.session_id, width_px=6, height_px=5)
single_img = decode_rgba(single_render.images[0].bytes_base64)

client.update_view(
    view_id=view_id,
    session_id=session.session_id,
    patch=[{"op": "replace", "path": "/view_2d/slice/slab", "value": {"thickness_vox": 5, "mode": "mip"}}],
)
mip_render = client.render_image(view_id=view_id, session_id=session.session_id, width_px=6, height_px=5)
mip_img = decode_rgba(mip_render.images[0].bytes_base64)

client.update_view(
    view_id=view_id,
    session_id=session.session_id,
    patch=[{"op": "replace", "path": "/view_2d/slice/slab", "value": {"thickness_vox": 5, "mode": "mean"}}],
)
mean_render = client.render_image(view_id=view_id, session_id=session.session_id, width_px=6, height_px=5)
mean_img = decode_rgba(mean_render.images[0].bytes_base64)

single_value = int(single_img[2, 3, 0])
mip_value = int(mip_img[2, 3, 0])
mean_value = int(mean_img[2, 3, 0])
assert mip_value > mean_value > single_value

rendered_override = client.render_image(
    view_id=view_id,
    session_id=session.session_id,
    width_px=64,
    height_px=64,
    overrides_json_patch=[{"op": "replace", "path": "/selectors", "value": [{"axis": "z", "kind": "index", "index": 3, "clamp": True}]}],
)
after = client.get_view(view_id=view_id, session_id=session.session_id).view_state
assert after.state_version == single.view_state.state_version + 2
assert after.state_hash != rendered_override.state_hash

http_client.close()
{
    "schema_version": rendered_override.schema_version,
    "state_version_after": after.state_version,
    "slab_values": {"single": single_value, "mean": mean_value, "mip": mip_value},
}


{'schema_version': 1,
 'state_version_after': 9,
 'slab_values': {'single': 63, 'mean': 114, 'mip': 165}}

## Expected output

Verify the following checks pass when you run the notebook top-to-bottom:

- `schema_version == 1` in render responses
- PNG image MIME is `image/png`
- XY/XZ/YZ rendered shapes match requested dimensions
- Slab ordering check holds: `mip > mean > single` at the sampled pixel
- A render with `overrides_json_patch` changes render `state_hash` but does not mutate persisted view state version via `view/get`
